# 从排序算例到真实检索：SQLite FTS5 与证据生命周期

Python 3.10+、CPU、无模型密钥。使用 Python 自带 SQLite 的 FTS5 实际建索引与查询；语料是原创英文教学片段，不运行 embedding、reranker、LLM 或生产性能测试。若环境没有 FTS5，明确报错，不把普通字符串匹配冒充全文检索。

先预测三件事：Bob 的短文档占据全局第一名时，Alice 还能取到自己的证据吗？更新正文后，旧引用还能直接使用吗？没有匹配证据时，系统应输出什么？

依次运行，观察结果。最后把 `FILTER_BEFORE_LIMIT` 改为 False 再全量运行：这是故意引入“先截断，再过滤”的错误版本，报告会保留未通过项。比较两次报告，不把失败结果改成 True。

依据：[SQLite FTS5](https://www.sqlite.org/fts5.html)、[Qdrant filtering](https://qdrant.tech/documentation/search/filtering/)。核对 2026-10-15；算法/API 依据与本站教学输出分别记录。


In [ ]:
import sqlite3, re, hashlib, json, tempfile
from pathlib import Path

FILTER_BEFORE_LIMIT = True  # 改 False，预测并观察召回缺失
lab_directory = tempfile.TemporaryDirectory(prefix='suguilin-rag-')
db_path = Path(lab_directory.name) / 'retrieval.sqlite'

def connect(path):
    db = sqlite3.connect(path)
    db.row_factory = sqlite3.Row
    return db

db = connect(db_path)
try:
    db.execute("CREATE VIRTUAL TABLE text_index USING fts5(chunk_id UNINDEXED, body, tokenize='unicode61')")
except sqlite3.OperationalError as error:
    raise RuntimeError('此 Python 的 SQLite 缺少 FTS5；请使用支持 FTS5 的 Python/Colab 环境') from error

db.execute('CREATE TABLE chunks (id TEXT PRIMARY KEY, tenant TEXT NOT NULL, version INTEGER NOT NULL, body TEXT NOT NULL)')

def put_chunk(db, chunk_id, tenant, version, body):
    if not chunk_id or not tenant or type(version) is not int or version < 1 or not body:
        raise ValueError('explicit ID, trusted tenant, positive version and body required')
    with db:
        old = db.execute('SELECT tenant,version FROM chunks WHERE id=?', (chunk_id,)).fetchone()
        if old and (old['tenant'] != tenant or version <= old['version']):
            raise ValueError('cross-tenant overwrite or non-increasing revision')
        db.execute('DELETE FROM text_index WHERE chunk_id=?', (chunk_id,))
        db.execute('INSERT INTO chunks VALUES (?,?,?,?) ON CONFLICT(id) DO UPDATE SET version=excluded.version, body=excluded.body', (chunk_id,tenant,version,body))
        db.execute('INSERT INTO text_index(chunk_id,body) VALUES (?,?)', (chunk_id,body))

def remove_chunk(db, chunk_id, tenant):
    with db:
        row = db.execute('SELECT tenant FROM chunks WHERE id=?', (chunk_id,)).fetchone()
        if row and row['tenant'] != tenant: raise PermissionError('tenant mismatch')
        db.execute('DELETE FROM text_index WHERE chunk_id=?', (chunk_id,))
        db.execute('DELETE FROM chunks WHERE id=? AND tenant=?', (chunk_id,tenant))

# 身份由测试夹具提供；真实服务必须从认证结果解析，不能接受模型/文档指定 tenant。
fixtures = [
    ('b1','bob',1,'checkpoint'),
    ('a1','alice',1,'checkpoint saves runtime state and supports safe resume'),
    ('a2','alice',1,'checkpoint durable workflow recovery'),
    ('a3','alice',1,'approval belongs to a particular draft revision'),
    ('b2','bob',1,'privatepayroll bob salary secret'),
    ('a4','alice',1,'vector similarity is not evidence of factual correctness'),
]
for record in fixtures: put_chunk(db,*record)
print('SQLite runtime:',sqlite3.sqlite_version, '| indexed chunks:',db.execute('SELECT count(*) FROM text_index').fetchone()[0])


## 1. 真的执行 MATCH 与 BM25，再看过滤的位置

这里把查询转为至多16个 ASCII 词项，每个词项加引号，以 OR 表达宽召回；不是自然语言查询改写。SQL 参数绑定与 FTS 查询语法是两个边界：参数绑定防止 SQL 注入，不会自动把用户输入变成合法 FTS 表达式。空查询、异常 k 与过长查询显式拒绝。

SQLite FTS5 的 bm25 值越小越好，按升序排列。这个共享小索引的词频统计仍来自全体文档，SQL 过滤只限制返回候选；它不是跨租户统计侧信道隔离或生产 ACL 认证。unicode61 也不是中文分词模型，换成连续中文需重新验证召回与分词协议。


In [ ]:
def fts_query(query):
    if not isinstance(query,str) or len(query)>512: raise ValueError('bounded text query required')
    raw_terms = re.findall(r'[a-z0-9_]+',query.lower())
    if not raw_terms or len(raw_terms)>16: raise ValueError('1..16 ASCII teaching terms required')
    terms = list(dict.fromkeys(raw_terms))
    return ' OR '.join('"'+term+'"' for term in terms)

def search(db,query,tenant,k=2,*,filter_before_limit=True):
    if type(k) is not int or not 1<=k<=20 or not isinstance(tenant,str) or not tenant:
        raise ValueError('bounded integer k and trusted tenant required')
    match=fts_query(query)
    columns='c.id,c.tenant,c.version,c.body,bm25(text_index) AS score'
    base=' FROM text_index JOIN chunks c ON c.id=text_index.chunk_id WHERE text_index MATCH ?'
    if filter_before_limit:
        sql='SELECT '+columns+base+' AND c.tenant=? ORDER BY score,c.id LIMIT ?'
        return [dict(row) for row in db.execute(sql,(match,tenant,k))]
    # 错误对照：全局 top-k 挤掉了有权访问的证据，随后过滤无法补回。
    sql='SELECT '+columns+base+' ORDER BY score,c.id LIMIT ?'
    return [dict(row) for row in db.execute(sql,(match,k)) if row['tenant']==tenant]

all_hits=[dict(row) for row in db.execute('SELECT c.id,bm25(text_index) AS score FROM text_index JOIN chunks c ON c.id=text_index.chunk_id WHERE text_index MATCH ? ORDER BY score,c.id',(fts_query('checkpoint'),))]
assert all_hits[0]['id']=='b1', all_hits
safe=search(db,'checkpoint','alice',1)
bad=search(db,'checkpoint','alice',1,filter_before_limit=False)
selected=search(db,'checkpoint','alice',1,filter_before_limit=FILTER_BEFORE_LIMIT)
assert len(safe)==1 and bad==[]
print('global ranking:',all_hits)
print('filter before top-1:',[r['id'] for r in safe],'| after top-1:',bad)
checks={'fts_ranking':all_hits[0]['score']<=all_hits[-1]['score'],
        'filter_before_limit':len(selected)==1 and selected[0]['tenant']=='alice'}

rejected=0
for args in [('', 'alice',2),('checkpoint','alice',True),('x '*17,'alice',2)]:
    try: search(db,*args)
    except ValueError: rejected+=1
checks['query_contract']=rejected==3 and search(db,'privatepayroll','alice')==[]


## 2. 更新、删除与旧引用

元数据表和 FTS 表在同一个 SQLite 事务里写入；这不自动保证远端向量库、对象存储或其他缓存也同步。引用记录 chunk ID、版本和正文摘要：正文发生变化后，即使 ID 相同，旧证据也不能作为当前版本继续使用。下面的校验只检查来源身份与版本，没有理解“引用是否支持某条自然语言结论”。


In [ ]:
def evidence(row):
    return {'id':row['id'],'version':row['version'],
            'sha256':hashlib.sha256(row['body'].encode()).hexdigest()}

def evidence_is_current(db,ref,tenant):
    row=db.execute('SELECT * FROM chunks WHERE id=? AND tenant=?',(ref['id'],tenant)).fetchone()
    return bool(row and evidence(row)==ref)

old=evidence(search(db,'approval','alice')[0])
put_chunk(db,'a3','alice',2,'authorization requires the latest draft revision')
checks['update_visibility']=search(db,'approval','alice')==[] and len(search(db,'authorization','alice'))==1
checks['citation_version']=not evidence_is_current(db,old,'alice') and evidence_is_current(db,evidence(search(db,'authorization','alice')[0]),'alice')
remove_chunk(db,'a3','alice')
checks['delete_visibility']=search(db,'authorization','alice')==[] and db.execute('SELECT count(*) FROM text_index WHERE chunk_id=?',('a3',)).fetchone()[0]==0
print('old citation rejected:',not evidence_is_current(db,old,'alice'),'| old/new terms removed:',checks['delete_visibility'])


## 3. 固定问题、标签、可见范围和检索指标

这些标签是人工构造的教学 gold，不是评测集成绩。正常问题计算证据 Recall；无答案问题 Recall 为 None，单独观察是否保持空证据。真实生成还要逐 claim 检查支持度、答案正确性和拒答准确性，本实验不声称完成这些测量。


In [ ]:
cases=[
    {'query':'checkpoint','tenant':'alice','gold':{'a1','a2'}},
    {'query':'factual correctness','tenant':'alice','gold':{'a4'}},
    {'query':'privatepayroll','tenant':'alice','gold':set()},
    {'query':'nonexistentterm','tenant':'alice','gold':set()},
]
observations=[]
for case in cases:
    hits=search(db,case['query'],case['tenant'],2,filter_before_limit=FILTER_BEFORE_LIMIT)
    ids={row['id'] for row in hits}
    recall=len(ids & case['gold'])/len(case['gold']) if case['gold'] else None
    observations.append({'query':case['query'],'ids':sorted(ids),'recall':recall,
                         'unauthorized':sum(row['tenant']!=case['tenant'] for row in hits)})
checks['evaluation_protocol']=observations[0]['recall']==1 and observations[1]['recall']==1 and all(row['unauthorized']==0 for row in observations)
empty=search(db,'nonexistentterm','alice')
retrieval_status={'status':'insufficient_evidence' if not empty else 'evidence_available','evidence':[evidence(row) for row in empty]}
checks['empty_evidence']=retrieval_status=={'status':'insufficient_evidence','evidence':[]}
db.close()
db=connect(db_path)
checks['sqlite_reopen']=len(search(db,'checkpoint','alice',2))==2
print(json.dumps(observations,ensure_ascii=False,indent=2))
print('checks:',checks)


## 4. 写出自己的解释，导回专题

先把下面的 `my_explanation` 改成自己的观察：错误版本丢掉了哪些证据、为什么过滤后不等于生成后的遮挡、更新与删除分别验证了什么。两个版本分别下载保存；导入站内后，未通过项用于提醒重做，不自动认定能力薄弱或安排掌握等级。

进一步实验：改变 k、增加同义改写、替换语料；固定标签再比较。向量检索与 reranker 是后续真实模型实验，不把这里的排序数组当作 embedding 输出。


In [ ]:
my_explanation = ''
report={'experiment':'rag-sqlite-retrieval','scope':'stdlib fixtures; no model/service',
        'checks':checks,'sqlite_version':sqlite3.sqlite_version,
        'filter_before_limit':FILTER_BEFORE_LIMIT,'observations':observations,
        'my_explanation':my_explanation}
Path('rag_retrieval_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved rag_retrieval_report.json; download it and import into the RAG topic.')
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download('rag_retrieval_report.json')
db.close()
lab_directory.cleanup()
